# Cricket Performance Analytics using PSL 2020-2021 Ball-by-Ball Dataset

University Data Analytics Capstone Project

This notebook walks through data cleaning, feature engineering, EDA, statistical analysis and visualization for the PSL 2020-2021 ball-by-ball dataset.

## Step 3: Data Cleaning and Preprocessing

In [2]:
"""
=====================================================================
01_data_cleaning.py
Cricket Performance Analytics - PSL 2020-2021
Step 3: Data Cleaning and Preprocessing
=====================================================================
Purpose:
    Load the raw ball-by-ball dataset, inspect its structure, handle
    missing/duplicate/inconsistent values, correct data types, and
    save a clean version of the dataset for downstream analysis.
"""

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

RAW_PATH = "../data/raw/psl_cleaned_2020_2021.csv"
CLEAN_PATH = "../data/cleaned/psl_cleaned_final.csv"

In [3]:

# ---------------------------------------------------------------
# 3.1 Load dataset
# ---------------------------------------------------------------
df = pd.read_csv(RAW_PATH)

print("=" * 70)
print("3.1 DATASET OVERVIEW")
print("=" * 70)
print(f"Shape (rows, columns): {df.shape}")
print("\nColumn names:")
print(list(df.columns))
print("\nFirst 5 rows:")
print(df.head())

3.1 DATASET OVERVIEW
Shape (rows, columns): (15164, 26)

Column names:
['id', 'match_id', 'date', 'season', 'venue', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder', 'winner', 'win_by', 'match_type', 'player_of_match', 'umpire_1', 'umpire_2']

First 5 rows:
   id  match_id        date  season             venue  inning      batting_team       bowling_team  over  ball    batter              bowler  \
0  92   1211642  2020-02-20    2020  National Stadium       1  Islamabad United  Quetta Gladiators     1     1   C Munro  Mohammad Nawaz (3)   
1  93   1211642  2020-02-20    2020  National Stadium       1  Islamabad United  Quetta Gladiators     1     2  DJ Malan  Mohammad Nawaz (3)   
2  94   1211642  2020-02-20    2020  National Stadium       1  Islamabad United  Quetta Gladiators     1     3  DJ Malan  Mohammad Nawaz (3)   
3  95   1211642  2020-0

In [4]:

# ---------------------------------------------------------------
# 3.2 Data types
# ---------------------------------------------------------------
print("\n" + "=" * 70)
print("3.2 DATA TYPES (before conversion)")
print("=" * 70)
print(df.dtypes)


3.2 DATA TYPES (before conversion)
id                  int64
match_id            int64
date                  str
season              int64
venue                 str
inning              int64
batting_team          str
bowling_team          str
over                int64
ball                int64
batter                str
bowler                str
non_striker           str
runs                int64
total_runs          int64
extras_type           str
is_wicket            bool
player_dismissed      str
dismissal_kind        str
fielder               str
winner                str
win_by                str
match_type            str
player_of_match       str
umpire_1              str
umpire_2              str
dtype: object


In [5]:

# NOTE: The source column for runs scored off the bat is named "runs".
# We rename it to "batsman_runs" to match the standard ball-by-ball
# cricket-analytics naming convention used throughout this project,
# and to avoid confusion with "total_runs" (which includes extras).
df = df.rename(columns={"runs": "batsman_runs"})

In [6]:
# ---------------------------------------------------------------
# 3.3 Missing values
# ---------------------------------------------------------------
print("\n" + "=" * 70)
print("3.3 MISSING VALUE CHECK")
print("=" * 70)
missing = df.isna().sum()
print(missing[missing > 0])


3.3 MISSING VALUE CHECK
player_of_match    102
dtype: int64


In [7]:
# player_of_match has 102 genuine missing values (likely abandoned /
# no-result matches where no Player of the Match was awarded).
# We keep these as NaN because they represent a real absence of an
# award, not a data-entry error.
print(f"\nRows with missing player_of_match: {df['player_of_match'].isna().sum()}")


Rows with missing player_of_match: 102


In [8]:
# ---------------------------------------------------------------
# 3.4 Structural "missing" values explanation
# ---------------------------------------------------------------
# The columns player_dismissed, dismissal_kind, extras_type and fielder
# are NOT missing in the conventional sense. The source data already
# encodes the "no event happened on this ball" case using explicit
# placeholder strings instead of NaN:
#   - extras_type       -> "No Extra"      when no extra was bowled
#   - player_dismissed  -> "Not Dismissed" when no wicket fell
#   - dismissal_kind    -> "Not Out"       when no wicket fell
#   - fielder           -> "No Fielder"    when no fielder was involved
#                                           (e.g. bowled, lbw, or no wicket)
# These are STRUCTURAL missing values: the absence of a value is itself
# meaningful information (most balls are NOT extras and NOT wickets),
# so they must never be dropped or imputed - they are analytically valid
# categories and are used directly in feature engineering (Step 4).
print("\nStructural placeholder categories confirmed:")
for col, placeholder in [
    ("extras_type", "No Extra"),
    ("player_dismissed", "Not Dismissed"),
    ("dismissal_kind", "Not Out"),
    ("fielder", "No Fielder"),
]:
    count = (df[col] == placeholder).sum()
    print(f"  {col:20s} -> '{placeholder}': {count} rows ({count/len(df)*100:.1f}%)")


Structural placeholder categories confirmed:
  extras_type          -> 'No Extra': 14337 rows (94.5%)
  player_dismissed     -> 'Not Dismissed': 14371 rows (94.8%)
  dismissal_kind       -> 'Not Out': 14371 rows (94.8%)
  fielder              -> 'No Fielder': 14626 rows (96.5%)


In [9]:

# ---------------------------------------------------------------
# 3.5 Duplicate check
# ---------------------------------------------------------------
print("\n" + "=" * 70)
print("3.5 DUPLICATE CHECK")
print("=" * 70)
dup_all = df.duplicated().sum()
dup_id = df.duplicated(subset=["id"]).sum()
print(f"Fully duplicated rows: {dup_all}")
print(f"Duplicated 'id' values: {dup_id}")
df = df.drop_duplicates()


3.5 DUPLICATE CHECK


Fully duplicated rows: 0
Duplicated 'id' values: 14845


In [10]:
# ---------------------------------------------------------------
# 3.6 Data type conversion
# ---------------------------------------------------------------
print("\n" + "=" * 70)
print("3.6 DATA TYPE CONVERSION")
print("=" * 70)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["season"] = df["season"].astype(int)
df["is_wicket"] = df["is_wicket"].astype(bool)

categorical_cols = [
    "venue", "batting_team", "bowling_team", "batter", "bowler",
    "non_striker", "extras_type", "player_dismissed", "dismissal_kind",
    "fielder", "winner", "match_type",
]
for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip()          # 3.7 remove extra spaces
    df[col] = df[col].astype("category")

print("Data types after conversion:")
print(df.dtypes)


3.6 DATA TYPE CONVERSION
Data types after conversion:
id                           int64
match_id                     int64
date                datetime64[us]
season                       int64
venue                     category
inning                       int64
batting_team              category
bowling_team              category
over                         int64
ball                         int64
batter                    category
bowler                    category
non_striker               category
batsman_runs                 int64
total_runs                   int64
extras_type               category
is_wicket                     bool
player_dismissed          category
dismissal_kind            category
fielder                   category
winner                    category
win_by                         str
match_type                category
player_of_match                str
umpire_1                       str
umpire_2                       str
dtype: object


In [11]:

# ---------------------------------------------------------------
# 3.7 Remove extra spaces / inconsistent text values
# ---------------------------------------------------------------
print("\n" + "=" * 70)
print("3.7 INCONSISTENT VALUE HANDLING")
print("=" * 70)


3.7 INCONSISTENT VALUE HANDLING


In [12]:
# Bowler names contain a trailing shirt-number in parentheses, e.g.
# "Mohammad Nawaz (3)". We keep a clean name column for player-level
# aggregation while preserving the original if ever needed.
df["bowler_clean"] = df["bowler"].astype(str).str.replace(
    r"\s*\(\d+\)\s*$", "", regex=True
).str.strip()

# Standardise venue text (strip repeated spaces, fix casing)
df["venue"] = df["venue"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

print("Example bowler name cleaning:")
print(df[["bowler", "bowler_clean"]].drop_duplicates().head(8))

Example bowler name cleaning:
                 bowler      bowler_clean
0    Mohammad Nawaz (3)    Mohammad Nawaz
6           Sohail Khan       Sohail Khan
26     Mohammad Hasnain  Mohammad Hasnain
32          BCJ Cutting       BCJ Cutting
38          Abdul Nasir       Abdul Nasir
44          Fawad Ahmed       Fawad Ahmed
119       Muhammad Musa     Muhammad Musa
125          Akif Javed        Akif Javed


In [13]:
# ---------------------------------------------------------------
# 3.8 Sanity checks on numeric ranges
# ---------------------------------------------------------------
print("\n" + "=" * 70)
print("3.8 NUMERIC RANGE SANITY CHECKS")
print("=" * 70)
print(f"over range: {df['over'].min()} - {df['over'].max()}")
print(f"ball range: {df['ball'].min()} - {df['ball'].max()}")
print(f"batsman_runs range: {df['batsman_runs'].min()} - {df['batsman_runs'].max()}")
print(f"total_runs range: {df['total_runs'].min()} - {df['total_runs'].max()}")
print(f"seasons present: {sorted(df['season'].unique())}")


3.8 NUMERIC RANGE SANITY CHECKS
over range: 1 - 20
ball range: 1 - 10
batsman_runs range: 0 - 6
total_runs range: 0 - 7
seasons present: [np.int64(2020), np.int64(2021)]


In [14]:


# ---------------------------------------------------------------
# 3.9 Save cleaned dataset
# ---------------------------------------------------------------
df.to_csv(CLEAN_PATH, index=False)
print("\n" + "=" * 70)
print(f"Cleaned dataset saved to: {CLEAN_PATH}")
print(f"Final shape: {df.shape}")
print("=" * 70)



Cleaned dataset saved to: ../data/cleaned/psl_cleaned_final.csv
Final shape: (15164, 27)


## Step 4: Feature Engineering

In [15]:
"""
=====================================================================
02_feature_engineering.py
Cricket Performance Analytics - PSL 2020-2021
Step 4: Feature Engineering
=====================================================================
Purpose:
    Derive analytically useful ball-level features from the cleaned
    dataset so that phase-wise, boundary, and wicket analysis can be
    performed efficiently in later steps.
"""

import pandas as pd
import numpy as np

df = pd.read_csv("../data/cleaned/psl_cleaned_final.csv")
df["date"] = pd.to_datetime(df["date"])

In [16]:


#----------------------------------------------
# 4.1 dot_ball
# Why: A dot ball (0 runs off the bat, no extras) shows bowling
# pressure/control. It is the base metric for run-rate & pressure
# analysis, and helps identify economical bowlers.
# ---------------------------------------------------------------
df["dot_ball"] = ((df["batsman_runs"] == 0) & (df["total_runs"] == 0)).astype(int)

In [17]:
# ---------------------------------------------------------------
# 4.2 four / six / boundary
# Why: Boundaries are the clearest indicator of attacking intent and
# batting power. Separating fours and sixes lets us compare players'
# and teams' hitting styles; "boundary" combines both for overall
# boundary-percentage metrics.
# ---------------------------------------------------------------
df["four"] = (df["batsman_runs"] == 4).astype(int)
df["six"] = (df["batsman_runs"] == 6).astype(int)
df["boundary"] = ((df["four"] == 1) | (df["six"] == 1)).astype(int)

In [18]:

# ---------------------------------------------------------------
# 4.3 wicket
# Why: Flags balls on which a wicket fell, independent of dismissal
# type. This underpins bowler strike-rate, team collapse and phase-
# wise wicket analysis.
# ---------------------------------------------------------------
df["wicket"] = df["is_wicket"].astype(int)

In [19]:
# ---------------------------------------------------------------
# 4.4 ball_count
# Why: A constant helper column (=1 per delivery) that makes
# groupby(...).sum() calls for "balls faced/bowled" trivial and
# readable, and is used as the denominator for strike-rate and
# economy-rate calculations.
# ---------------------------------------------------------------
df["ball_count"] = 1

In [20]:
# ---------------------------------------------------------------
# 4.4 ball_count
# Why: A constant helper column (=1 per delivery) that makes
# groupby(...).sum() calls for "balls faced/bowled" trivial and
# readable, and is used as the denominator for strike-rate and
# economy-rate calculations.
# ---------------------------------------------------------------
df["ball_count"] = 1

In [21]:

# ---------------------------------------------------------------
# 4.5 powerplay / middle_over / death_over
# Why: T20 innings are strategically divided into three phases with
# different risk/run profiles. Tagging each ball by phase is essential
# for phase-wise scoring-rate, wicket, and strategy analysis.
#   Powerplay   : overs 1-6   (field restrictions, attacking starts)
#   Middle overs: overs 7-15  (rebuilding / rotation phase)
#   Death overs : overs 16-20 (aggressive finishing overs)
# ---------------------------------------------------------------
df["powerplay"] = df["over"].between(1, 6).astype(int)
df["middle_over"] = df["over"].between(7, 15).astype(int)
df["death_over"] = df["over"].between(16, 20).astype(int)


def phase_label(over):
    if 1 <= over <= 6:
        return "Powerplay"
    elif 7 <= over <= 15:
        return "Middle Overs"
    else:
        return "Death Overs"


df["match_phase"] = df["over"].apply(phase_label)

In [22]:

# ---------------------------------------------------------------
# 4.6 run_type
# Why: A single categorical label for every delivery's scoring
# outcome (dot, single/double/triple, four, six, or extra) simplifies
# distribution and pie-chart style analysis of how runs are scored.
# ---------------------------------------------------------------


def classify_run(row):
    if row["is_wicket"] and row["total_runs"] == 0:
        return "Wicket"
    if row["batsman_runs"] == 6:
        return "Six"
    if row["batsman_runs"] == 4:
        return "Four"
    if row["batsman_runs"] in (1, 2, 3):
        return "Running Runs"
    if row["batsman_runs"] == 0 and row["total_runs"] > 0:
        return "Extra"
    return "Dot Ball"


df["run_type"] = df.apply(classify_run, axis=1)

In [23]:

# ---------------------------------------------------------------
# 4.7 Additional helper features used across the EDA section
# ---------------------------------------------------------------
df["is_extra"] = (df["extras_type"] != "No Extra").astype(int)
df["extra_runs"] = df["total_runs"] - df["batsman_runs"]
df["match_year"] = df["date"].dt.year
df["match_month"] = df["date"].dt.to_period("M").astype(str)

# Save feature-engineered dataset
OUT_PATH = "../data/cleaned/psl_features.csv"
df.to_csv(OUT_PATH, index=False)

print("Feature engineering complete.")
print(f"New shape: {df.shape}")
print("New columns added:")
new_cols = [
    "dot_ball", "four", "six", "boundary", "wicket", "ball_count",
    "powerplay", "middle_over", "death_over", "match_phase", "run_type",
    "is_extra", "extra_runs", "match_year", "match_month",
]
print(new_cols)
print(f"\nSaved to: {OUT_PATH}")
print("\nrun_type distribution:")
print(df["run_type"].value_counts())


Feature engineering complete.
New shape: (15164, 42)
New columns added:
['dot_ball', 'four', 'six', 'boundary', 'wicket', 'ball_count', 'powerplay', 'middle_over', 'death_over', 'match_phase', 'run_type', 'is_extra', 'extra_runs', 'match_year', 'match_month']

Saved to: ../data/cleaned/psl_features.csv

run_type distribution:
run_type
Running Runs    6513
Dot Ball        4372
Four            1887
Six              827
Extra            797
Wicket           768
Name: count, dtype: int64


## Steps 5-7: EDA, Statistical Analysis & 20 Visualizations

In [24]:
"""
=====================================================================
03_eda_statistics_visuals.py
Cricket Performance Analytics - PSL 2020-2021
Steps 5, 6 & 7: EDA, Statistical Analysis, 20 Visualizations
=====================================================================
All 20 charts are saved as PNG files into ../images/
A text summary of every numeric result is printed to stdout and also
captured to ../reports/analysis_output_log.txt for use in the report.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 130
plt.rcParams["savefig.bbox"] = "tight"

IMG = "../images/"
df = pd.read_csv("../data/cleaned/psl_features.csv")
df["date"] = pd.to_datetime(df["date"])

PALETTE = "viridis"
TEAM_PALETTE = "Set2"

# =====================================================================
# helper: match-level table (one row per match) - needed for win/venue stats
# =====================================================================
match_df = df.drop_duplicates(subset="match_id")[
    ["match_id", "date", "season", "venue", "winner", "win_by",
     "match_type", "player_of_match"]
].reset_index(drop=True)

In [25]:
# team total runs per match/innings (for run distribution per match)
team_innings = (
    df.groupby(["match_id", "inning", "batting_team"])["total_runs"]
    .sum().reset_index()
)

print("#" * 70)
print("STEP 5: EXPLORATORY DATA ANALYSIS (EDA)")
print("#" * 70)

######################################################################
STEP 5: EXPLORATORY DATA ANALYSIS (EDA)
######################################################################


In [26]:
# ---------------------------------------------------------------
# 5.1 Dataset Summary
# ---------------------------------------------------------------
print("\n--- 5.1 DATASET SUMMARY ---")
print(f"Total deliveries (rows): {len(df)}")
print(f"Total matches: {df['match_id'].nunique()}")
print(f"Total teams: {df['batting_team'].nunique()}")
print(f"Total unique batters: {df['batter'].nunique()}")
print(f"Total unique bowlers: {df['bowler_clean'].nunique()}")
print(f"Total venues: {df['venue'].nunique()}")
print(f"Seasons covered: {sorted(df['season'].unique())}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")


--- 5.1 DATASET SUMMARY ---
Total deliveries (rows): 15164
Total matches: 66
Total teams: 6
Total unique batters: 157
Total unique bowlers: 106
Total venues: 6
Seasons covered: [np.int64(2020), np.int64(2021)]
Date range: 2020-02-20 to 2021-06-24


In [27]:
# ---------------------------------------------------------------
# 5.2 Descriptive Statistics
# ---------------------------------------------------------------
print("\n--- 5.2 DESCRIPTIVE STATISTICS (numeric columns) ---")
desc = df[["batsman_runs", "total_runs", "extra_runs", "over", "ball"]].describe()
print(desc)


--- 5.2 DESCRIPTIVE STATISTICS (numeric columns) ---
       batsman_runs    total_runs    extra_runs          over          ball
count  15164.000000  15164.000000  15164.000000  15164.000000  15164.000000
mean       1.318715      1.388288      0.069573      9.983514      3.621604
std        1.686907      1.673426      0.355064      5.628464      1.814317
min        0.000000      0.000000      0.000000      1.000000      1.000000
25%        0.000000      0.000000      0.000000      5.000000      2.000000
50%        1.000000      1.000000      0.000000     10.000000      4.000000
75%        1.000000      1.000000      0.000000     15.000000      5.000000
max        6.000000      7.000000      5.000000     20.000000     10.000000


In [28]:
# ---------------------------------------------------------------
# 6. STATISTICAL ANALYSIS (mean, median, mode, std, var, corr, cov, skew, kurtosis)
# ---------------------------------------------------------------
print("\n" + "#" * 70)
print("STEP 6: STATISTICAL ANALYSIS")
print("#" * 70)

for col in ["batsman_runs", "total_runs"]:
    s = df[col]
    print(f"\nColumn: {col}")
    print(f"  Mean     : {s.mean():.4f}")
    print(f"  Median   : {s.median():.4f}")
    print(f"  Mode     : {s.mode().iloc[0]}")
    print(f"  Std Dev  : {s.std():.4f}")
    print(f"  Variance : {s.var():.4f}")
    print(f"  Skewness : {s.skew():.4f}")
    print(f"  Kurtosis : {s.kurtosis():.4f}")

corr_matrix = df[["batsman_runs", "total_runs", "extra_runs", "is_wicket",
                   "over", "ball", "dot_ball", "boundary"]].astype(float).corr()
print("\nCorrelation matrix:")
print(corr_matrix.round(3))

cov_val = df["batsman_runs"].astype(float).cov(df["is_wicket"].astype(float))
print(f"\nCovariance(batsman_runs, is_wicket): {cov_val:.4f}")


######################################################################
STEP 6: STATISTICAL ANALYSIS
######################################################################

Column: batsman_runs
  Mean     : 1.3187
  Median   : 1.0000
  Mode     : 0
  Std Dev  : 1.6869
  Variance : 2.8457
  Skewness : 1.4852
  Kurtosis : 1.1867

Column: total_runs
  Mean     : 1.3883
  Median   : 1.0000
  Mode     : 1
  Std Dev  : 1.6734
  Variance : 2.8004
  Skewness : 1.4590
  Kurtosis : 1.1425

Correlation matrix:
              batsman_runs  total_runs  extra_runs  is_wicket   over   ball  dot_ball  boundary
batsman_runs         1.000       0.978      -0.143     -0.180  0.078 -0.012    -0.559     0.911
total_runs           0.978       1.000       0.068     -0.190  0.082 -0.013    -0.594     0.900
extra_runs          -0.143       0.068       1.000     -0.043  0.016 -0.007    -0.140    -0.084
is_wicket           -0.180      -0.190      -0.043      1.000  0.062  0.013     0.313    -0.110
over           

In [29]:

# =====================================================================
# CATEGORICAL / NUMERICAL / VENUE / PLAYER / TEAM / SEASON / PHASE / etc.
# =====================================================================
print("\n--- 5.3 CATEGORICAL ANALYSIS: Teams ---")
print(df["batting_team"].value_counts())

print("\n--- 5.4 VENUE ANALYSIS ---")
venue_matches = match_df.groupby("venue")["match_id"].nunique().sort_values(ascending=False)
print(venue_matches)

print("\n--- 5.5 SEASON ANALYSIS: Runs per season ---")
season_runs = df.groupby("season")["total_runs"].sum()
print(season_runs)

print("\n--- 5.6 PLAYER ANALYSIS: Top 10 run scorers ---")
top_scorers = df.groupby("batter")["batsman_runs"].sum().sort_values(ascending=False).head(10)
print(top_scorers)

print("\n--- 5.7 PLAYER ANALYSIS: Top 10 wicket takers ---")
top_wicket_takers = df[df["is_wicket"] == True].groupby("bowler_clean")["is_wicket"].sum() \
    .sort_values(ascending=False).head(10)
print(top_wicket_takers)

print("\n--- 5.8 TEAM ANALYSIS: Match wins ---")
team_wins = match_df["winner"].value_counts()
print(team_wins)

print("\n--- 5.9 POWERPLAY / MIDDLE / DEATH OVER ANALYSIS ---")
phase_runs = df.groupby("match_phase")["total_runs"].sum()
phase_wickets = df.groupby("match_phase")["is_wicket"].sum()
phase_balls = df.groupby("match_phase")["ball_count"].sum()
phase_rr = (phase_runs / phase_balls * 6).round(2)
print("Runs per phase:\n", phase_runs)
print("Wickets per phase:\n", phase_wickets)
print("Run-rate per phase:\n", phase_rr)

print("\n--- 5.10 BOUNDARY ANALYSIS ---")
print(f"Total fours: {df['four'].sum()}  | Total sixes: {df['six'].sum()}")
print("Top 10 boundary hitters:")
print(df.groupby("batter")["boundary"].sum().sort_values(ascending=False).head(10))

print("\n--- 5.11 PLAYER OF THE MATCH ANALYSIS ---")
print(match_df["player_of_match"].value_counts().head(10))

print("\n--- 5.12 MATCH TYPE ANALYSIS ---")
print(match_df["match_type"].value_counts())

print("\n--- 5.13 TIME-SERIES: Matches per month ---")
matches_month = match_df.copy()
matches_month["month"] = matches_month["date"].dt.to_period("M").astype(str)
print(matches_month.groupby("month")["match_id"].nunique())


--- 5.3 CATEGORICAL ANALYSIS: Teams ---
batting_team
Multan Sultans       2666
Karachi Kings        2629
Lahore Qalandars     2627
Peshawar Zalmi       2620
Islamabad United     2384
Quetta Gladiators    2238
Name: count, dtype: int64

--- 5.4 VENUE ANALYSIS ---
venue
National Stadium                   25
Sheikh Zayed Stadium, Abu Dhabi    20
Gaddafi Stadium                    10
Rawalpindi Cricket Stadium          7
Multan Cricket Stadium              3
National Stadium, Karachi           1
Name: match_id, dtype: int64

--- 5.5 SEASON ANALYSIS: Runs per season ---
season
2020     9899
2021    11153
Name: total_runs, dtype: int64

--- 5.6 PLAYER ANALYSIS: Top 10 run scorers ---
batter
Babar Azam         1027
Shoaib Malik        632
Fakhar Zaman        612
Mohammad Hafeez     583
Sharjeel Khan       555
Kamran Akmal        534
C Munro             533
Mohammad Rizwan     500
Shan Masood         492
Sarfraz Ahmed       469
Name: batsman_runs, dtype: int64

--- 5.7 PLAYER ANALYSIS: Top 10

In [30]:


# =====================================================================
# STEP 7: 20 PROFESSIONAL VISUALIZATIONS
# =====================================================================
print("\n" + "#" * 70)
print("STEP 7: GENERATING 20 VISUALIZATIONS")
print("#" * 70)

# 1. Top Run Scorers
plt.figure(figsize=(9, 6))
top_scorers.sort_values().plot(kind="barh", color=sns.color_palette(PALETTE, 10))
plt.title("Top 10 Run Scorers - PSL 2020-2021", fontsize=14, fontweight="bold")
plt.xlabel("Total Runs")
plt.ylabel("Batter")
plt.tight_layout()
plt.savefig(IMG + "01_top_run_scorers.png")
plt.close()


######################################################################
STEP 7: GENERATING 20 VISUALIZATIONS
######################################################################


In [31]:

# =====================================================================
# CATEGORICAL / NUMERICAL / VENUE / PLAYER / TEAM / SEASON / PHASE / etc.
# =====================================================================
print("\n--- 5.3 CATEGORICAL ANALYSIS: Teams ---")
print(df["batting_team"].value_counts())

print("\n--- 5.4 VENUE ANALYSIS ---")
venue_matches = match_df.groupby("venue")["match_id"].nunique().sort_values(ascending=False)
print(venue_matches)

print("\n--- 5.5 SEASON ANALYSIS: Runs per season ---")
season_runs = df.groupby("season")["total_runs"].sum()
print(season_runs)

print("\n--- 5.6 PLAYER ANALYSIS: Top 10 run scorers ---")
top_scorers = df.groupby("batter")["batsman_runs"].sum().sort_values(ascending=False).head(10)
print(top_scorers)

print("\n--- 5.7 PLAYER ANALYSIS: Top 10 wicket takers ---")
top_wicket_takers = df[df["is_wicket"] == True].groupby("bowler_clean")["is_wicket"].sum() \
    .sort_values(ascending=False).head(10)
print(top_wicket_takers)

print("\n--- 5.8 TEAM ANALYSIS: Match wins ---")
team_wins = match_df["winner"].value_counts()
print(team_wins)

print("\n--- 5.9 POWERPLAY / MIDDLE / DEATH OVER ANALYSIS ---")
phase_runs = df.groupby("match_phase")["total_runs"].sum()
phase_wickets = df.groupby("match_phase")["is_wicket"].sum()
phase_balls = df.groupby("match_phase")["ball_count"].sum()
phase_rr = (phase_runs / phase_balls * 6).round(2)
print("Runs per phase:\n", phase_runs)
print("Wickets per phase:\n", phase_wickets)
print("Run-rate per phase:\n", phase_rr)

print("\n--- 5.10 BOUNDARY ANALYSIS ---")
print(f"Total fours: {df['four'].sum()}  | Total sixes: {df['six'].sum()}")
print("Top 10 boundary hitters:")
print(df.groupby("batter")["boundary"].sum().sort_values(ascending=False).head(10))

print("\n--- 5.11 PLAYER OF THE MATCH ANALYSIS ---")
print(match_df["player_of_match"].value_counts().head(10))

print("\n--- 5.12 MATCH TYPE ANALYSIS ---")
print(match_df["match_type"].value_counts())

print("\n--- 5.13 TIME-SERIES: Matches per month ---")
matches_month = match_df.copy()
matches_month["month"] = matches_month["date"].dt.to_period("M").astype(str)
print(matches_month.groupby("month")["match_id"].nunique())


--- 5.3 CATEGORICAL ANALYSIS: Teams ---
batting_team
Multan Sultans       2666
Karachi Kings        2629
Lahore Qalandars     2627
Peshawar Zalmi       2620
Islamabad United     2384
Quetta Gladiators    2238
Name: count, dtype: int64

--- 5.4 VENUE ANALYSIS ---
venue
National Stadium                   25
Sheikh Zayed Stadium, Abu Dhabi    20
Gaddafi Stadium                    10
Rawalpindi Cricket Stadium          7
Multan Cricket Stadium              3
National Stadium, Karachi           1
Name: match_id, dtype: int64

--- 5.5 SEASON ANALYSIS: Runs per season ---
season
2020     9899
2021    11153
Name: total_runs, dtype: int64

--- 5.6 PLAYER ANALYSIS: Top 10 run scorers ---
batter
Babar Azam         1027
Shoaib Malik        632
Fakhar Zaman        612
Mohammad Hafeez     583
Sharjeel Khan       555
Kamran Akmal        534
C Munro             533
Mohammad Rizwan     500
Shan Masood         492
Sarfraz Ahmed       469
Name: batsman_runs, dtype: int64

--- 5.7 PLAYER ANALYSIS: Top 10

In [32]:
# =====================================================================
#  Scorers
plt.figure(figsize=(9, 6))
top_scorers.sort_values().plot(kind="barh", color=sns.color_palette(PALETTE, 10))
plt.title("Top 10 Run Scorers - PSL 2020-2021", fontsize=14, fontweight="bold")
plt.xlabel("Total Runs")
plt.ylabel("Batter")
plt.tight_layout()
plt.savefig(IMG + "01_top_run_scorers.png")
# =====================================================================
print("\n" + "#" * 70)
print("STEP 7: GENERATING 20 VISUALIZATIONS")
print("#" * 70)

# 1. Top Run Scorers
plt.close()


######################################################################
STEP 7: GENERATING 20 VISUALIZATIONS
######################################################################


In [33]:
# 2. Top Wicket Takers
plt.figure(figsize=(9, 6))
top_wicket_takers.sort_values().plot(kind="barh", color=sns.color_palette("mako", 10))
plt.title("Top 10 Wicket Takers - PSL 2020-2021", fontsize=14, fontweight="bold")
plt.xlabel("Total Wickets")
plt.ylabel("Bowler")
plt.tight_layout()
plt.savefig(IMG + "02_top_wicket_takers.png")
plt.close()



In [34]:
# 3. Team-wise Runs
team_runs = df.groupby("batting_team")["total_runs"].sum().sort_values(ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(x=team_runs.values, y=team_runs.index, palette=TEAM_PALETTE)
plt.title("Team-wise Total Runs Scored - PSL 2020-2021", fontsize=14, fontweight="bold")
plt.xlabel("Total Runs")
plt.ylabel("Team")
plt.tight_layout()
plt.savefig(IMG + "03_team_wise_runs.png")
plt.close()


C:\Users\HP\AppData\Local\Temp\ipykernel_17452\3740367357.py:4: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=team_runs.values, y=team_runs.index, palette=TEAM_PALETTE)


In [35]:

# 4. Team-wise Wins
plt.figure(figsize=(10, 6))
sns.barplot(x=team_wins.values, y=team_wins.index, palette=TEAM_PALETTE)
plt.title("Team-wise Match Wins - PSL 2020-2021", fontsize=14, fontweight="bold")
plt.xlabel("Number of Wins")
plt.ylabel("Team")
plt.tight_layout()
plt.savefig(IMG + "04_team_wise_wins.png")
plt.close()

C:\Users\HP\AppData\Local\Temp\ipykernel_17452\3235955635.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=team_wins.values, y=team_wins.index, palette=TEAM_PALETTE)


In [36]:


# 5. Top Boundary Hitters
top_boundary = df.groupby("batter")["boundary"].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(9, 6))
top_boundary.sort_values().plot(kind="barh", color=sns.color_palette("crest", 10))
plt.title("Top 10 Boundary Hitters (4s + 6s) - PSL 2020-2021", fontsize=14, fontweight="bold")
plt.xlabel("Total Boundaries")
plt.tight_layout()
plt.savefig(IMG + "05_top_boundary_hitters.png")
plt.close()

In [37]:
# 6. Top Six Hitters
top_six = df.groupby("batter")["six"].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(9, 6))
top_six.sort_values().plot(kind="barh", color=sns.color_palette("flare", 10))
plt.title("Top 10 Six Hitters - PSL 2020-2021", fontsize=14, fontweight="bold")
plt.xlabel("Total Sixes")
plt.tight_layout()
plt.savefig(IMG + "06_top_six_hitters.png")
plt.close()

In [38]:
# 7. Player of the Match Awards
pom = match_df["player_of_match"].value_counts().head(10)
plt.figure(figsize=(9, 6))
pom.sort_values().plot(kind="barh", color=sns.color_palette("rocket", 10))
plt.title("Top 10 Player of the Match Award Winners", fontsize=14, fontweight="bold")
plt.xlabel("Awards Won")
plt.tight_layout()
plt.savefig(IMG + "07_player_of_match.png")
plt.close()


In [39]:
# 8. Team-wise Wickets (wickets taken by bowling team)
team_wickets = df[df["is_wicket"] == True].groupby("bowling_team")["is_wicket"].sum() \
    .sort_values(ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(x=team_wickets.values, y=team_wickets.index, palette="Set2")
plt.title("Team-wise Wickets Taken - PSL 2020-2021", fontsize=14, fontweight="bold")
plt.xlabel("Wickets Taken")
plt.tight_layout()
plt.savefig(IMG + "08_team_wise_wickets.png")
plt.close()

C:\Users\HP\AppData\Local\Temp\ipykernel_17452\2424028073.py:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=team_wickets.values, y=team_wickets.index, palette="Set2")


In [40]:
# 9. Venue-wise Matches
plt.figure(figsize=(10, 6))
sns.barplot(x=venue_matches.values, y=venue_matches.index, palette="crest")
plt.title("Number of Matches Played per Venue", fontsize=14, fontweight="bold")
plt.xlabel("Matches Played")
plt.tight_layout()
plt.savefig(IMG + "09_venue_wise_matches.png")
plt.close()

C:\Users\HP\AppData\Local\Temp\ipykernel_17452\3337545337.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=venue_matches.values, y=venue_matches.index, palette="crest")


In [41]:
# 10. Venue-wise Average Runs (per innings)
venue_avg_runs = df.groupby(["venue", "match_id", "inning"])["total_runs"] \
    .sum().reset_index().groupby("venue")["total_runs"].mean().sort_values(ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(x=venue_avg_runs.values, y=venue_avg_runs.index, palette="mako")
plt.title("Average Runs per Innings by Venue", fontsize=14, fontweight="bold")
plt.xlabel("Average Runs per Innings")
plt.tight_layout()
plt.savefig(IMG + "10_venue_avg_runs.png")
plt.close()


C:\Users\HP\AppData\Local\Temp\ipykernel_17452\1283741294.py:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=venue_avg_runs.values, y=venue_avg_runs.index, palette="mako")


In [42]:

# 11. Runs per Over (Line Chart)
runs_per_over = df.groupby("over")["total_runs"].sum() / df["match_id"].nunique() / 2
# average runs scored in that over across all innings
runs_per_over_avg = df.groupby(["match_id", "inning", "over"])["total_runs"].sum() \
    .reset_index().groupby("over")["total_runs"].mean()
plt.figure(figsize=(10, 6))
plt.plot(runs_per_over_avg.index, runs_per_over_avg.values, marker="o", color="#21295C", linewidth=2)
plt.title("Average Runs Scored per Over - PSL 2020-2021", fontsize=14, fontweight="bold")
plt.xlabel("Over Number")
plt.ylabel("Average Runs")
plt.xticks(range(1, 21))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(IMG + "11_runs_per_over.png")
plt.close()

In [43]:


# 12. Wickets per Over (Line Chart)
wickets_per_over_avg = df.groupby(["match_id", "inning", "over"])["is_wicket"].sum() \
    .reset_index().groupby("over")["is_wicket"].mean()
plt.figure(figsize=(10, 6))
plt.plot(wickets_per_over_avg.index, wickets_per_over_avg.values, marker="o", color="#990011", linewidth=2)
plt.title("Average Wickets Lost per Over - PSL 2020-2021", fontsize=14, fontweight="bold")
plt.xlabel("Over Number")
plt.ylabel("Average Wickets")
plt.xticks(range(1, 21))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(IMG + "12_wickets_per_over.png")
plt.close()

In [44]:


# 13. Powerplay vs Middle vs Death Runs
plt.figure(figsize=(8, 6))
phase_order = ["Powerplay", "Middle Overs", "Death Overs"]
sns.barplot(x=phase_order, y=[phase_runs[p] for p in phase_order],
            palette=["#028090", "#00A896", "#02C39A"])
plt.title("Total Runs by Match Phase - PSL 2020-2021", fontsize=14, fontweight="bold")
plt.ylabel("Total Runs")
plt.tight_layout()
plt.savefig(IMG + "13_phase_runs.png")
plt.close()

C:\Users\HP\AppData\Local\Temp\ipykernel_17452\4237992500.py:4: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=phase_order, y=[phase_runs[p] for p in phase_order],


In [45]:
# 14. Distribution of Runs (Histogram)
plt.figure(figsize=(9, 6))
sns.histplot(df["batsman_runs"], bins=7, kde=False, color="#B85042")
plt.title("Distribution of Runs Scored per Ball", fontsize=14, fontweight="bold")
plt.xlabel("Runs off the Bat")
plt.ylabel("Frequency (balls)")
plt.tight_layout()
plt.savefig(IMG + "14_runs_distribution.png")
plt.close()

In [46]:
# 15. Correlation Heatmap
plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Heatmap of Key Ball-Level Metrics", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(IMG + "15_correlation_heatmap.png")
plt.close()


In [47]:
# 16. Scatter Plot: Team total runs vs wickets lost per innings
inn_summary = df.groupby(["match_id", "inning", "batting_team"]).agg(
    total_runs=("total_runs", "sum"), wickets=("is_wicket", "sum")
).reset_index()
plt.figure(figsize=(9, 6))
sns.scatterplot(data=inn_summary, x="wickets", y="total_runs", hue="wickets",
                 palette="viridis", legend=False, s=70, alpha=0.75)
plt.title("Innings Total Runs vs Wickets Lost", fontsize=14, fontweight="bold")
plt.xlabel("Wickets Lost in Innings")
plt.ylabel("Total Runs Scored in Innings")
plt.tight_layout()
plt.savefig(IMG + "16_runs_vs_wickets_scatter.png")
plt.close()

In [48]:

# 17. Box Plot: Runs per over distribution by phase
overs_summary = df.groupby(["match_id", "inning", "over"]).agg(
    total_runs=("total_runs", "sum")
).reset_index()
overs_summary["match_phase"] = overs_summary["over"].apply(phase_label := (
    lambda o: "Powerplay" if o <= 6 else ("Middle Overs" if o <= 15 else "Death Overs")
))
plt.figure(figsize=(9, 6))
sns.boxplot(data=overs_summary, x="match_phase", y="total_runs", order=phase_order,
            palette=["#F96167", "#F9E795", "#2F3C7E"])
plt.title("Distribution of Runs per Over by Match Phase", fontsize=14, fontweight="bold")
plt.xlabel("Match Phase")
plt.ylabel("Runs in the Over")
plt.tight_layout()
plt.savefig(IMG + "17_boxplot_runs_by_phase.png")
plt.close()

C:\Users\HP\AppData\Local\Temp\ipykernel_17452\1133836252.py:9: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=overs_summary, x="match_phase", y="total_runs", order=phase_order,


In [49]:
# 18. Violin Plot: Runs per over by phase
plt.figure(figsize=(9, 6))
sns.violinplot(data=overs_summary, x="match_phase", y="total_runs", order=phase_order,
                palette=["#84B59F", "#69A297", "#50808E"])
plt.title("Violin Plot: Runs per Over Distribution by Phase", fontsize=14, fontweight="bold")
plt.xlabel("Match Phase")
plt.ylabel("Runs in the Over")
plt.tight_layout()
plt.savefig(IMG + "18_violin_runs_by_phase.png")
plt.close()

C:\Users\HP\AppData\Local\Temp\ipykernel_17452\1381608238.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=overs_summary, x="match_phase", y="total_runs", order=phase_order,


In [50]:

# 19. Pie Chart: Overall run type share
run_type_counts = df["run_type"].value_counts()
plt.figure(figsize=(8, 8))
colors = sns.color_palette("Set3", len(run_type_counts))
plt.pie(run_type_counts.values, labels=run_type_counts.index, autopct="%1.1f%%",
        startangle=90, colors=colors)
plt.title("Share of Delivery Outcomes (Run Type) - PSL 2020-2021", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(IMG + "19_run_type_pie.png")
plt.close()

In [51]:

# 20. Season-wise Performance Comparison
season_summary = df.groupby("season").agg(
    total_runs=("total_runs", "sum"),
    total_wickets=("is_wicket", "sum"),
    total_sixes=("six", "sum"),
    total_fours=("four", "sum"),
).reset_index()
season_summary_melt = season_summary.melt(id_vars="season",
                                           value_vars=["total_runs", "total_wickets",
                                                       "total_sixes", "total_fours"])
plt.figure(figsize=(10, 6))
sns.barplot(data=season_summary_melt, x="variable", y="value", hue="season", palette="Set1")
plt.title("Season-wise Performance Comparison (2020 vs 2021)", fontsize=14, fontweight="bold")
plt.xlabel("Metric")
plt.ylabel("Total Count")
plt.legend(title="Season")
plt.tight_layout()
plt.savefig(IMG + "20_season_comparison.png")
plt.close()

print("\nAll 20 visualizations saved successfully to ../images/")
print("\nSeason summary table:")
print(season_summary)

print("\n" + "#" * 70)
print("EDA, STATISTICAL ANALYSIS AND VISUALIZATION SCRIPT COMPLETE")
print("#" * 70)



All 20 visualizations saved successfully to ../images/

Season summary table:
   season  total_runs  total_wickets  total_sixes  total_fours
0    2020        9899            375          387          875
1    2021       11153            418          440         1012

######################################################################
EDA, STATISTICAL ANALYSIS AND VISUALIZATION SCRIPT COMPLETE
######################################################################


In [1]:
# cd "E:\Python\Practice\Cricket Parformance Analyst\Cricket_Performance_Analytics"
# py -m streamlit run app.py